In [2]:
# ライブラリ

In [3]:
import os
import sys
import glob
from IPython.display import display
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
#import japanize_matplotlib
import seaborn as sns 
import sweetviz as sv
import yaml

sys.path.append(os.path.abspath('..'))
from configs.config import *

In [4]:
import importlib
import configs.config
importlib.reload(configs.config)
from configs.config import *

In [5]:
pd.set_option("display.max_columns",100)
pd.set_option("display.max_rows", 500)

# データ読み込み

In [6]:
df_train = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_TRAIN))
df_test = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_TEST))
df_sample_submission = pd.read_csv(os.path.join(DIR_INPUT, FILE_SAMPLE_SUBMISSION))
df_udemy_activity = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_UDEMY_ACTIVITY))
df_career = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_CAREER))
df_dx = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_DX))
df_hr = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_HR))
df_overtime_work_by_month = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_ORVER_TIME))
df_position_history = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_POSITION_HISTORY))

In [7]:
print(f"train shape: {df_train.shape}")
print(f"test shape: {df_test.shape}")  
print(f"sample submission shape: {df_sample_submission.shape}")
print(f"udemy activity shape: {df_udemy_activity.shape}")
print(f"career shape: {df_career.shape}")
print(f"dx shape: {df_dx.shape}")
print(f"hr shape: {df_hr.shape}")
print(f"overtime work by month shape: {df_overtime_work_by_month.shape}")
print(f"position history shape: {df_position_history.shape}")

train shape: (7338, 3)
test shape: (11022, 2)
sample submission shape: (11022, 1)
udemy activity shape: (539164, 12)
career shape: (375, 46)
dx shape: (7100, 4)
hr shape: (7076, 4)
overtime work by month shape: (101439, 3)
position history shape: (8596, 4)


# 型、項目数、欠損値率

In [8]:
def calc_describe(df):
    dtypes = []
    val_counts_station = []
    isnull_station = []
    isnull_station_ratio = 100 * df.isnull().sum() / len(df)
    for col in df.columns:
        dtypes.append(str(df[col].dtype))
        val_counts_station.append(len(df[col].value_counts()))
        isnull_station.append(isnull_station_ratio[col])
    inds = ["型", "val_counts", "NaN率"]
    df_eda = pd.DataFrame([dtypes, val_counts_station, isnull_station], columns=df.columns, index=inds).T
    # df_eda.query("val_counts > 1")
    return df_eda

In [9]:
# 各データフレームのEDAを実行
for df, name in zip([df_train, df_test, df_sample_submission, df_udemy_activity, df_career, df_dx, df_hr, df_overtime_work_by_month, df_position_history],
             ["df_train", "df_test", "df_sample_submission", "df_udemy_activity", "df_career", "df_dx", "df_hr", "df_overtime_work_by_month", "df_position_history"]):
    print(f"{name}")
    df_eda = calc_describe(df)
    display(df_eda)

df_train


,型,val_counts,NaN率
社員番号,object,1223,0.0
category,object,6,0.0
target,int64,2,0.0


df_test


,型,val_counts,NaN率
社員番号,object,1837,0.0
category,object,6,0.0


df_sample_submission


...

,型,val_counts,NaN率
社員番号,object,1550,0.0
カテゴリ,object,12,0.0
研修名,object,19,16.308649
実施日,object,767,0.0


df_overtime_work_by_month


,型,val_counts,NaN率
社員番号,object,3060,0.0
date,object,36,0.0
hours,float64,175,0.0


df_position_history


,型,val_counts,NaN率
社員番号,object,3060,0.0
year,int64,3,0.0
勤務区分,object,2,0.0
役職,object,10,0.0


# 基本統計量

In [10]:
for df, name in zip([df_train, df_test, df_sample_submission, df_udemy_activity, df_career, df_dx, df_hr, df_overtime_work_by_month, df_position_history],
             ["df_train", "df_test", "df_sample_submission", "df_udemy_activity", "df_career", "df_dx", "df_hr", "df_overtime_work_by_month", "df_position_history"]):
    print(f"{name}")
    display(df.head())

df_train


,社員番号,category,target
0,-2Sq3E0WkZj8pL7jxdL3Cg==,コンテンツ・サービス・デザイン,0
1,-2Sq3E0WkZj8pL7jxdL3Cg==,コーポレート管理部門/技術・データ・BPR,0
2,-2Sq3E0WkZj8pL7jxdL3Cg==,プロダクトマネジメント,0
3,-2Sq3E0WkZj8pL7jxdL3Cg==,マーケティング,0
4,-2Sq3E0WkZj8pL7jxdL3Cg==,事業企画・開発・研究,0


df_test


,社員番号,category
0,-1sqs0GXzpPJuAVKHUUFgg==,コンテンツ・サービス・デザイン
1,-1sqs0GXzpPJuAVKHUUFgg==,コーポレート管理部門/技術・データ・BPR
2,-1sqs0GXzpPJuAVKHUUFgg==,プロダクトマネジメント
3,-1sqs0GXzpPJuAVKHUUFgg==,マーケティング
4,-1sqs0GXzpPJuAVKHUUFgg==,事業企画・開発・研究


df_sample_submission


...

,社員番号,カテゴリ,研修名,実施日
0,-4taxxVbT1nU-J5fHWmDfQ==,3等級昇格者研修,3等級昇格者研修,2023-06-02 00:00:00
1,-4taxxVbT1nU-J5fHWmDfQ==,ビジネススキルアップ研修,マーケティング,"2022/11/15,2022/11/29"
2,-4taxxVbT1nU-J5fHWmDfQ==,ビジネススキルアップ研修,マーケティング,"2022/11/15,2022/11/29"
3,-4taxxVbT1nU-J5fHWmDfQ==,ビジネススキルアップ研修,マーケティング,"2022/11/15,2022/11/29"
4,-4taxxVbT1nU-J5fHWmDfQ==,ビジネススキルアップ研修,マーケティング,"2022/11/15,2022/11/29"


df_overtime_work_by_month


,社員番号,date,hours
0,-1sqs0GXzpPJuAVKHUUFgg==,2022-01-01,10.0
1,-1sqs0GXzpPJuAVKHUUFgg==,2022-02-01,11.0
2,-1sqs0GXzpPJuAVKHUUFgg==,2022-03-01,18.0
3,-1sqs0GXzpPJuAVKHUUFgg==,2022-04-01,30.0
4,-1sqs0GXzpPJuAVKHUUFgg==,2022-05-01,27.0


df_position_history


,社員番号,year,勤務区分,役職
0,-1sqs0GXzpPJuAVKHUUFgg==,22,正社員,一般
1,-1sqs0GXzpPJuAVKHUUFgg==,23,正社員,一般
2,-1sqs0GXzpPJuAVKHUUFgg==,24,正社員,グループリーダー
3,-2Sq3E0WkZj8pL7jxdL3Cg==,23,正社員,一般
4,-2Sq3E0WkZj8pL7jxdL3Cg==,24,正社員,一般


In [11]:
for df, name in zip([df_train, df_test, df_sample_submission, df_udemy_activity, df_career, df_dx, df_hr, df_overtime_work_by_month, df_position_history],
             ["df_train", "df_test", "df_sample_submission", "df_udemy_activity", "df_career", "df_dx", "df_hr", "df_overtime_work_by_month", "df_position_history"]):
    print(f"{name}")
    display(df.describe())

df_train


,target
count,7338.000000
mean,0.039384
std,0.194520
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


df_test


,社員番号,category
count,11022,11022
unique,1837,6
top,-1sqs0GXzpPJuAVKHUUFgg==,コンテンツ・サービス・デザイン
freq,6,1837


df_sample_submission


...

,社員番号,カテゴリ,研修名,実施日
count,7076,7076,5922,7076
unique,1550,12,19,767
top,M-J4K8nv5EWsiGlan6HRNA==,ビジネススキルアップ研修,アカウンティング,2024-12-01 00:00:00
freq,34,2940,918,316


df_overtime_work_by_month


,hours
count,101439.000000
mean,38.600883
std,22.332754
min,0.000000
25%,23.000000
50%,39.000000
75%,54.000000
max,195.000000


df_position_history


,year
count,8596.000000
mean,23.066892
std,0.800417
min,22.000000
25%,22.000000
50%,23.000000
75%,24.000000
max,24.000000


# train

In [32]:
df_train

,社員番号,category,target
0,-2Sq3E0WkZj8pL7jxdL3Cg==,コンテンツ・サービス・デザイン,0
1,-2Sq3E0WkZj8pL7jxdL3Cg==,コーポレート管理部門/技術・データ・BPR,0
2,-2Sq3E0WkZj8pL7jxdL3Cg==,プロダクトマネジメント,0
3,-2Sq3E0WkZj8pL7jxdL3Cg==,マーケティング,0
4,-2Sq3E0WkZj8pL7jxdL3Cg==,事業企画・開発・研究,0
...,...,...,...
7333,zuplFpzBoM4c1dFy5HPXqg==,コーポレート管理部門/技術・データ・BPR,0
7334,zuplFpzBoM4c1dFy5HPXqg==,プロダクトマネジメント,0
7335,zuplFpzBoM4c1dFy5HPXqg==,マーケティング,0
7336,zuplFpzBoM4c1dFy5HPXqg==,事業企画・開発・研究,0


In [35]:
df_train['category'].value_counts()

category
コンテンツ・サービス・デザイン          1223
コーポレート管理部門/技術・データ・BPR    1223
プロダクトマネジメント              1223
マーケティング                  1223
事業企画・開発・研究               1223
営業                       1223
Name: count, dtype: int64

In [14]:
df_train.groupby('category')['target'].value_counts().reset_index()

,category,target,count
0,コンテンツ・サービス・デザイン,0,1193
1,コンテンツ・サービス・デザイン,1,30
2,コーポレート管理部門/技術・データ・BPR,0,1161
3,コーポレート管理部門/技術・データ・BPR,1,62
4,プロダクトマネジメント,0,1162
5,プロダクトマネジメント,1,61
6,マーケティング,0,1171
7,マーケティング,1,52
8,事業企画・開発・研究,0,1167
9,事業企画・開発・研究,1,56


# df_test

In [15]:
df_test.head()

,社員番号,category
0,-1sqs0GXzpPJuAVKHUUFgg==,コンテンツ・サービス・デザイン
1,-1sqs0GXzpPJuAVKHUUFgg==,コーポレート管理部門/技術・データ・BPR
2,-1sqs0GXzpPJuAVKHUUFgg==,プロダクトマネジメント
3,-1sqs0GXzpPJuAVKHUUFgg==,マーケティング
4,-1sqs0GXzpPJuAVKHUUFgg==,事業企画・開発・研究


In [34]:
df_test['category'].value_counts()

category
コンテンツ・サービス・デザイン          1837
コーポレート管理部門/技術・データ・BPR    1837
プロダクトマネジメント              1837
マーケティング                  1837
事業企画・開発・研究               1837
営業                       1837
Name: count, dtype: int64

# udemy_activity

In [16]:
df_udemy_activity

,社員番号,コースID,コースタイトル,レクチャーもしくはクイズ,レクチャー/クイズID,レクチャー/クイズの題名,開始日,終了日,推定完了率%,最終結果（クイズの場合）,マーク済み修了,コースカテゴリー
0,-1sqs0GXzpPJuAVKHUUFgg==,4615016,企業オリジナル講座,Quiz,5528090,企業オリジナル講座,2022/4/11 10:10,2022/4/11 10:10,100.0,0.0,True,企業オリジナル講座
1,-1sqs0GXzpPJuAVKHUUFgg==,4615016,企業オリジナル講座,Quiz,5528090,企業オリジナル講座,2022/4/11 10:11,2022/4/11 10:11,100.0,100.0,True,企業オリジナル講座
2,-1sqs0GXzpPJuAVKHUUFgg==,4615016,企業オリジナル講座,Quiz,5528100,企業オリジナル講座,2022/4/11 10:26,2022/4/11 10:26,100.0,0.0,True,企業オリジナル講座
3,-1sqs0GXzpPJuAVKHUUFgg==,4615016,企業オリジナル講座,Quiz,5528100,企業オリジナル講座,2022/4/11 10:27,2022/4/11 10:27,100.0,100.0,True,企業オリジナル講座
4,-1sqs0GXzpPJuAVKHUUFgg==,4615016,企業オリジナル講座,Quiz,5528102,企業オリジナル講座,2022/4/11 10:21,2022/4/11 10:21,100.0,100.0,True,企業オリジナル講座
...,...,...,...,...,...,...,...,...,...,...,...,...
539159,zxY0Eflwm1tYj1Wt6vo_1g==,5264112,企業オリジナル講座,Video_lecture,37311144,企業オリジナル講座,2023/6/14 9:56,2023/6/14 9:56,100.0,NaN,True,企業オリジナル講座
539160,zxY0Eflwm1tYj1Wt6vo_1g==,5264112,企業オリジナル講座,Video_lecture,37311150,企業オリジナル講座,2023/6/14 9:57,2023/6/14 9:57,100.0,NaN,True,企業オリジナル講座
539161,zxY0Eflwm1tYj1Wt6vo_1g==,6098315,企業オリジナル講座,Video_lecture,45012691,企業オリジナル講座,2024/9/27 15:39,2024/9/27 16:17,100.0,NaN,True,企業オリジナル講座
539162,zxY0Eflwm1tYj1Wt6vo_1g==,6106205,企業オリジナル講座,Video_lecture,45088199,企業オリジナル講座,2024/12/16 16:21,2024/12/16 17:39,100.0,NaN,True,企業オリジナル講座


In [17]:
df_udemy_activity.dtypes

社員番号             object
コースID             int64
コースタイトル          object
レクチャーもしくはクイズ     object
レクチャー/クイズID       int64
レクチャー/クイズの題名     object
開始日              object
終了日              object
推定完了率%          float64
最終結果（クイズの場合）    float64
マーク済み修了            bool
コースカテゴリー         object
dtype: object

In [38]:
df_udemy_activity['コースタイトル'].value_counts().reset_index()

,コースタイトル,count
0,企業オリジナル講座,42743
1,ITパスポート最速合格コース ～効率的な学習で0から合格まで～,11800
2,【ChatGPTで仕事を10倍効率化】AIを味方につけてデキる人材になる！ビジネスに使える1...,8516
3,【旧講座】今日から始めるデジタルトランスフォーメーション！テクノロジーの仕組みからデータ活用...,6556
4,未経験者でもスッキリ分かる！初心者のためのプロダクトマネージャー超入門,5203
...,...,...
3432,"Master Discrete Mathematics: Sets, Math Logic,...",1
3433,④社労士受験講座 NO4：雇用保険法「右脳を使って効率学習！イメージマスター社労士講座 20...,1
3434,⑦社労士受験講座 NO７：国民年金法「右脳を使って効率学習！イメージマスター社労士講座 20...,1
3435,Certified Minitab Beginner: Graphical Tools (A...,1


In [40]:
df_udemy_activity['レクチャーもしくはクイズ'].value_counts().reset_index()

,レクチャーもしくはクイズ,count
0,Video_lecture,491743
1,Other_lecture,25032
2,Quiz,22389


In [41]:
df_udemy_activity['レクチャー/クイズの題名'].value_counts().reset_index()

,レクチャー/クイズの題名,count
0,企業オリジナル講座,42743
1,このセクションで学ぶこと,5585
2,はじめに,5039
3,このセクションのまとめ,4030
4,まとめ,2129
...,...,...
85018,ITシステム構成,1
85019,Part.3-3 デモ,1
85020,セッションとCookie,1
85021,HTML概要,1


In [42]:
df_udemy_activity['コースカテゴリー'].value_counts().reset_index()

,コースカテゴリー,count
0,ビジネス戦略,46497
1,企業オリジナル講座,42743
2,IT資格,39244
3,オフィスの生産性向上とコラボレーションツール,38302
4,ビジネス向け生成AI,24513
5,プロジェクト管理,19770
6,マーケティング戦略,12883
7,チームマネジメント,12573
8,プログラミング言語,12021
9,ウェブ開発,11471


# career

In [18]:
df_career.head()

,社員番号,自分の能力を発揮できる仕事上の得意分野が見つかっている\n,自分はどんな仕事をやりたいのか明らかである \n,自分は何を望んで今の仕事をしているのかわかっている\n,自分なりの職業的な生き方に関する目標・目的がはっきりしている\n,自分のこれからのキャリアには、あまり関心がない\n,これからのキャリアを、より充実したものにしたいと強く思う\n,キャリア設計（職業生活の設計）は、自分にとって重要な課題である\n,これからのキャリアをどう歩むべきか、あまり考えていない\n,納得いくキャリアを歩めるかどうかは、自分の責任だと思う\n,キャリア形成は、自分自身の責任である\n,納得いくキャリアを歩めない原因の大半は周囲の環境にある\n,キャリアは周りの環境によって決められていくものだと思う\n,新しい環境や状況にも、わりあい早くなじんで対応している,職場環境の変化に対して自分なりに考えて対応している,新しい職場に移ってもすぐに自分らしさを発揮している,職場の制度や仕事が変わってもすぐ対応している,自分の職種・業界分野における最新動向を常に情報収集している,仕事のために新しいことをいろいろ勉強している,社会・経済の動きや成り行きに関する情報を、積極的に収集している,新しい知識・技術を積極的に学ぶように努めている,新しい人間関係が構築できるように、社内外の活動に積極的に参加している,仕事と直接関係ない人とも積極的に交流するようにしている,新しいネットワークづくりに常に取り組んでいる,自分の満足感を高めるように、仕事のやり方を工夫している,常に自発的に仕事を行っている,自分の価値観やポリシーを持って仕事に取り組んでいる,仕事の進め方や企画を立てる上で、今までの延長上のやり方ではなく、自分なりの発想を持って取り組んでいる,少人数チームで協力してタスクを完遂できる,メンバー全員のアイデアを取り入れて成果物を作れる,同僚や上司に向けて口頭でプレゼンテーションができる,スライドや動画などのデジタル資料を用いて情報を共有できる,異なる視点を比較して最適な解決策を選べる,正解が一つでない問題に対して解決策を導き出せる,複数のアイデアを試行し改善を重ねられる,困難な業務課題に対して独自の解決策を考案できる,自分の進捗をモニタリングし計画を柔軟に調整できる,フィードバックを受けて自ら仕事を改善できる,業務データを可視化し BI ダッシュボードで意思決定に活用できる,RPA／ノーコードツールで反復的な業務フローを自動化できる,生成 AI や機械学習モデルを業務改善に活用できる,クラウド環境のアクセス権限を監査し情報保護を徹底できる,デジタルマーケティング施策（SNS 広告等）をデータで最適化できる,顧客・市場データを用いて新商品・サービス企画を立案できる,クラウド／DevOps 環境を構築し IaC で継続的デプロイを管理できる,デジタルツインや IoT データを用いたサプライチェーン最適化を提案・実装できる
0,-4taxxVbT1nU-J5fHWmDfQ==,4 そう思う／当てはまる,3 どちらとも言えない,4 そう思う／当てはまる,3 どちらとも言えない,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,3 どちらとも言えない,4 そう思う／当てはまる,3 どちらとも言えない,3 どちらとも言えない,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,3 どちらとも言えない,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,3 どちらとも言えない,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,2 あまりそう思わない／あまり当てはまらない,4 そう思う／当てはまる,3 どちらとも言えない,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,5 強くそう思う／とても当てはまる,3 どちらとも言えない,2 あまりそう思わない／あまり当てはまらない,3 どちらとも言えない,3 どちらとも言えない,2 あまりそう思わない／あまり当てはまらない,2 あまりそう思わない／あまり当てはまらない,2 あまりそう思わない／あまり当てはまらない,2 あまりそう思わない／あまり当てはまらない
1,-EtuCRccKFQgi3UfXRvRkA==,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,1 全くそう思わない／全く当てはまらない,1 全くそう思わない／全く当てはまらない,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,1 全くそう思わない／全く当てはまらない,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,1 全くそう思わない／全く当てはまらない,1 全くそう思わない／全く当てはまらない,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,1 全くそう思わない／全く当てはまらない,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,1 全くそう思わない／全く当てはまらない
2,-JxDfe1I3lMhJDvo7mmvoA==,4 そう思う／当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,1 全くそう思わない／全く当てはまらない,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,1 全くそう思わない／全く当てはまらない,4 そう思う／当てはまる,4 そう思う／当てはまる,1 全くそう思わない／全く当てはまらない,1 全くそう思わない／全く当てはまらない,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,2 あまりそう思わない／あまり当てはまらない,2 あまりそう思わない／あまり当てはまらない,2 あまりそう思わない／あまり当てはまらない,4 そう思う／当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,3 どちらとも言えない,4 そう思う／当てはまる,2 あまりそう思わない／あまり当てはまらない,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,2 あまりそう思わない／あまり当てはまらない,2 あまりそう思わない／あまり当てはまらない
3,-Q5JF_Zj03QFrpbCJWPR3A==,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,1 全くそう思わない／全く当てはまらない,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,1 全くそう思わない／全く当てはまらない,1 全くそう思わない／全く当てはまらない,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,5 強くそう思う／とても当てはまる,4 そう思う／当てはまる,4 そう思う／当てはまる,5 

# dx

In [19]:
df_dx

,社員番号,研修実施日,研修カテゴリ,研修名
0,-2Sq3E0WkZj8pL7jxdL3Cg==,2024-04-15 00:00:00,リテラシー_DX基礎,【DX基礎研修】DX概論
1,-2Sq3E0WkZj8pL7jxdL3Cg==,2024-04-16 00:00:00,リテラシー_DX基礎,【DX基礎研修】ものづくり概論
2,-5W_JQCSTAYe2gGJMuT4_w==,2023-04-20 00:00:00,リテラシー_DX基礎,【DX基礎研修】社内事例①
3,-5W_JQCSTAYe2gGJMuT4_w==,2023-04-20 00:00:00,リテラシー_DX基礎,【DX基礎研修】社内事例②
4,-5W_JQCSTAYe2gGJMuT4_w==,2024-04-15 00:00:00,リテラシー_DX基礎,【DX基礎研修】イントロダクション
...,...,...,...,...
7095,zuplFpzBoM4c1dFy5HPXqg==,2024-10-04 00:00:00,データ利活用_実践,【DP13】テキストマイニングPython20241004
7096,zuplFpzBoM4c1dFy5HPXqg==,2024-10-23 00:00:00,データ利活用_実践,【DP07】モダンExcel PowerBI1023
7097,zxPKKLM85QljzRfp0yisow==,2023-08-23 00:00:00,データ利活用_実践,事業データと活用研修
7098,zxY0Eflwm1tYj1Wt6vo_1g==,2023-09-28 00:00:00,DX講演会,先生DX講演会


In [20]:
df_dx['研修カテゴリ'].value_counts()

研修カテゴリ
リテラシー_DX基礎        4252
DX講演会             1301
データ利活用_実践          640
プロダクト開発_基礎         342
デジタルマーケティング_基礎     316
データ利活用_基礎          149
データ                 35
プロダクト開発_実践          24
システム開発_実践           23
DX勉強会               18
Name: count, dtype: int64

In [21]:
df_dx['研修名'].value_counts()

研修名
【DX基礎研修】DX概論                           453
【DX基礎研修】イントロダクション                      430
コミュニケーション学 「ツッコミ講座」                    333
【DX基礎研修】デジタルマーケ事例① インフルエンサーマーケティング     306
【DX基礎研修】ディスラプター②                       249
【DX基礎研修】ディスラプター基礎                      219
【DX基礎研修】社内事例④                          218
【LB00-02】DX講演会 教育に生成AI 1129            213
先生DX講演会                                206
【DX基礎研修】ディスラプター事例                      201
「デジタル革命の先にある新しい社会」                     182
【DX基礎研修】ものづくり事例③                       176
【DX基礎研修】社内事例②                          170
【DX基礎研修】ものづくり事例①                       169
【DX基礎研修】ものづくり事例②                       168
【DX基礎研修】DIF基礎                          158
【DX基礎研修】コンプライアンス・セキュリティ                153
【DX基礎研修】ものづくり概論 UX概論 UIUX実践例・工夫例       150
【DX基礎研修】社内事例①                          149
【DX基礎研修】社内事例③                          146
アフターデジタル時代のUX経営                        146
【DX基礎研修】デジタルマーケ事例② CDP                 137
【DX基礎研修】DIF事例① コアとなっている技術や思想とサービス紹介    131
IT３社講演会

# hr

In [22]:
df_hr

,社員番号,カテゴリ,研修名,実施日
0,-4taxxVbT1nU-J5fHWmDfQ==,3等級昇格者研修,3等級昇格者研修,2023-06-02 00:00:00
1,-4taxxVbT1nU-J5fHWmDfQ==,ビジネススキルアップ研修,マーケティング,"2022/11/15,2022/11/29"
2,-4taxxVbT1nU-J5fHWmDfQ==,ビジネススキルアップ研修,マーケティング,"2022/11/15,2022/11/29"
3,-4taxxVbT1nU-J5fHWmDfQ==,ビジネススキルアップ研修,マーケティング,"2022/11/15,2022/11/29"
4,-4taxxVbT1nU-J5fHWmDfQ==,ビジネススキルアップ研修,マーケティング,"2022/11/15,2022/11/29"
...,...,...,...,...
7071,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"
7072,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"
7073,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"
7074,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"


In [23]:
df_hr[df_hr['実施日']=='2024/5/17,2024/7/83']

,社員番号,カテゴリ,研修名,実施日
25,-WNGlH_XHI3_niAnslX8gg==,マネジメント研修,新任課長研修,"2024/5/17,2024/7/83"
26,-WNGlH_XHI3_niAnslX8gg==,マネジメント研修,新任課長研修,"2024/5/17,2024/7/83"


In [24]:
df_hr[df_hr['社員番号']=='zxmtr2h4ypvsNq02K9AMJg==']

,社員番号,カテゴリ,研修名,実施日
7070,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"
7071,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"
7072,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"
7073,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"
7074,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"
7075,zxmtr2h4ypvsNq02K9AMJg==,ビジネススキルアップ研修,管理会計,"2023/10/23,2023/11/13"


In [25]:
df_hr['実施日'].value_counts()

実施日
2024-12-01 00:00:00    316
2024-06-06 00:00:00    306
2023-07-07 00:00:00    252
2023-09-08 00:00:00    250
2023/7/10,2023/7/31    216
                      ... 
2024-07-02 00:00:00      1
2024-04-12 00:00:00      1
2022-10-05 00:00:00      1
2022-12-06 00:00:00      1
2022-11-05 00:00:00      1
Name: count, Length: 767, dtype: int64

# overtime_work_by_month

In [ ]:
df_overtime_work_by_month

,社員番号,date,hours
0,-1sqs0GXzpPJuAVKHUUFgg==,2022-01-01,10.0
1,-1sqs0GXzpPJuAVKHUUFgg==,2022-02-01,11.0
2,-1sqs0GXzpPJuAVKHUUFgg==,2022-03-01,18.0
3,-1sqs0GXzpPJuAVKHUUFgg==,2022-04-01,30.0
4,-1sqs0GXzpPJuAVKHUUFgg==,2022-05-01,27.0
...,...,...,...
101434,zxmtr2h4ypvsNq02K9AMJg==,2024-08-01,34.0
101435,zxmtr2h4ypvsNq02K9AMJg==,2024-09-01,34.0
101436,zxmtr2h4ypvsNq02K9AMJg==,2024-10-01,33.0
101437,zxmtr2h4ypvsNq02K9AMJg==,2024-11-01,26.0


# position_history

In [27]:
df_position_history

,社員番号,year,勤務区分,役職
0,-1sqs0GXzpPJuAVKHUUFgg==,22,正社員,一般
1,-1sqs0GXzpPJuAVKHUUFgg==,23,正社員,一般
2,-1sqs0GXzpPJuAVKHUUFgg==,24,正社員,グループリーダー
3,-2Sq3E0WkZj8pL7jxdL3Cg==,23,正社員,一般
4,-2Sq3E0WkZj8pL7jxdL3Cg==,24,正社員,一般
...,...,...,...,...
8591,zxY0Eflwm1tYj1Wt6vo_1g==,23,正社員,一般
8592,zxY0Eflwm1tYj1Wt6vo_1g==,24,正社員,一般
8593,zxmtr2h4ypvsNq02K9AMJg==,22,正社員,一般
8594,zxmtr2h4ypvsNq02K9AMJg==,23,正社員,一般


In [31]:
df_position_history.dtypes

社員番号    object
year     int64
勤務区分    object
役職      object
dtype: object

In [29]:
df_position_history['勤務区分'].value_counts()

勤務区分
正社員         7251
正社員(管理職)    1345
Name: count, dtype: int64

In [30]:
df_position_history['役職'].value_counts()

役職
一般          6653
グループリーダー     897
課長           686
部長           181
副部長           66
副本部長          34
室長            32
本部長           29
支社長           13
副室長            5
Name: count, dtype: int64

# 前処理済データの読み込み

In [5]:
df_prep_station = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_station.pkl"))
df_prep_status = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_status.pkl"))
df_prep_trip = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_trip.pkl"))
df_prep_weather = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_weather.pkl"))